In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
import json
from pathlib import Path
import warnings
from scipy import stats
warnings.filterwarnings('ignore')

# Set style
# plt.style.use('default')
plt.style.use("ggplot")
sns.set_palette("husl")

# Set figure parameters
plt.rcParams['figure.figsize'] = (12, 8)
plt.rcParams['font.size'] = 12

In [2]:
STATIC_POOL_DIR = "final"
LEARN_POOL_DIR = "final_2"

In [6]:
def load_data(pool_dir):
    # Get all strategy directories
    strategy_dirs = [d for d in pool_dir.iterdir() if d.is_dir() and not d.name.startswith('analysis')]
    strategy_dirs = sorted(strategy_dirs, key=lambda x: int(x.name.split('_')[0]))

    print(f"Found {len(strategy_dirs)} strategy directories:")
    for d in strategy_dirs:
        print(f"  {d.name}")

    # Get noise levels from one directory
    sample_rep_dir = strategy_dirs[0] / 'repetition_0'
    csv_files = [f for f in sample_rep_dir.iterdir() if f.suffix == '.csv']
    noise_levels = sorted([float(f.stem) for f in csv_files])
    print(f"\nNoise levels: {noise_levels}")

    # Number of repetitions
    rep_dirs = [d for d in strategy_dirs[0].iterdir() if d.name.startswith('repetition_')]
    n_repetitions = len(rep_dirs)
    print(f"Number of repetitions: {n_repetitions}")

    def load_tournament_data():
        """
        Load all tournament data from the final directory structure.
        Returns a comprehensive DataFrame with all results.
        """
        all_data = []
        
        for strategy_dir in strategy_dirs:
            strategy_id = int(strategy_dir.name.split('_')[0])
            strategy_name = '_'.join(strategy_dir.name.split('_')[1:])
            
            print(f"Loading data for {strategy_dir.name}...")
            
            for rep in range(n_repetitions):
                rep_dir = strategy_dir / f'repetition_{rep}'
                
                # Load player names from details.json
                details_file = rep_dir / 'details.json'
                if details_file.exists():
                    with open(details_file, 'r') as f:
                        details = json.load(f)
                        player_names = details['player_names']
                else:
                    player_names = None
                
                for noise_level in noise_levels:
                    csv_file = rep_dir / f'{noise_level}.csv'
                    
                    if csv_file.exists():
                        try:
                            df = pd.read_csv(csv_file)
                            
                            # Add metadata columns
                            df['strategy_id'] = strategy_id
                            df['strategy_name'] = strategy_name
                            df['testing_strategy'] = f"{strategy_id}_{strategy_name}"
                            df['noise_level'] = noise_level
                            df['repetition_id'] = rep

                            # calculate last N-steps cc_rate 
                            actions = df['Actions']
                            print(actions.shape)
                            return
                            # Add player names if available
                            if player_names:
                                df['player_name_clean'] = df['Player index'].map(
                                    lambda x: player_names[x] if x < len(player_names) else f"Player_{x}"
                                )
                                df['opponent_name_clean'] = df['Opponent index'].map(
                                    lambda x: player_names[x] if x < len(player_names) else f"Player_{x}"
                                )
                            
                            all_data.append(df)
                            
                        except Exception as e:
                            print(f"Error loading {csv_file}: {e}")
        
        # Combine all data
        full_df = pd.concat(all_data, ignore_index=True)
        print(f"\nLoaded {len(full_df)} total interactions")
        
        return full_df
    tournament_data = load_tournament_data()
    return tournament_data

load_data(Path(STATIC_POOL_DIR))

Found 18 strategy directories:
  0_DBS
  1_RiskyQLearner
  2_ArrogantQLearner
  3_CautiousQLearner
  4_HesitantQLearner
  5_JaxFiveStateAgent
  6_JaxFiveStateAgent
  7_JaxFactorizedAgent
  8_JaxFactorizedAgent
  9_APavlov2011
  10_AdaptiveTitForTat
  11_GTFT
  12_ContriteTitForTat
  13_StochasticWSLS
  14_WinStayLoseShift
  15_Cooperator
  16_Defector
  17_JaxFiveStateAgent

Noise levels: [0.0, 0.05, 0.1, 0.15, 0.2, 0.25, 0.3, 0.35, 0.4, 0.45]
Number of repetitions: 10
Loading data for 0_DBS...
(342,)
